# <b><span style='color:#F1A424'>|</span> CS506: <span style='color:#F1A424'>Hypothyroidism Prediction Project</span><span style='color:#ABABAB'> [Advance]</span></b> 

***

### <b><span style='color:#F1A424'>Table of Contents</span></b> <a class='anchor' id='top'></a>
<div style=" background-color:#3b3745; padding: 13px 13px; border-radius: 8px; color: white">
    <li> <a href="#install_libraries">Install libraries</a></li>
    <li> <a href="#introduction">Introduction</a></li>
    <li><a href="#import_libraries">Import Libraries</a></li>
    <li><a href="#configuration">Configuration</a></li>
    <li><a href="#utils">Utils</a></li>
    <li><a href="#load_data">Load Data</a></li>
    <li><a href="#preprocessing">Data Pre-processing</a></li>
    <li><a href="#validation">Validation</a></li>
    <li><a href="#modeling">Modeling</a></li>
</div>


# <b><span style='color:#F1A424'>|</span> Introduction</b><a class='anchor' id='introduction'></a> [↑](#top) 

***



# <b><span style='color:#F1A424'>|</span> Install Libraries</b><a class='anchor' id='install_libraries'></a> [↑](#top) 

***

Install all the required libraries for this notebook.

# <b><span style='color:#F1A424'>|</span> Import Libraries</b><a class='anchor' id='import_libraries'></a> [↑](#top) 

***

Import all the required libraries for this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import sklearn
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

from glob import glob

from sklearn.preprocessing import LabelEncoder

# <b><span style='color:#F1A424'>|</span> Configuration</b><a class='anchor' id='configuration'></a> [↑](#top) 

***

Configuration for this notebook.

In [ ]:
class paths:
    OUTPUT = "/kaggle/working"
    TRAIN_CSV = "/kaggle/input/datasets/apssg96/uc-irvine-thyroid-disease/allhypo.data"
    TEST_CSV = "/kaggle/input/datasets/apssg96/uc-irvine-thyroid-disease/allhypo.test"

# <b><span style='color:#F1A424'>|</span> Utils</b><a class='anchor' id='utils'></a> [↑](#top) 

***

Utils for this notebook.

In [ ]:
def sep():
    print("_"*100)

# <b><span style='color:#F1A424'>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

***

Load data for this notebook.

In [ ]:
COLS_ALL = [
    "age", "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication",
    "sick", "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid",
    "query_hyperthyroid", "lithium", "goitre", "tumor", "hypopituitary", "psych",
    "TSH_measured", "TSH", "T3_measured", "T3", "TT4_measured", "TT4", "T4U_measured",
    "T4U", "FTI_measured", "FTI", "TBG_measured", "TBG", "referral_source",
    "target"
]
COLS_CATEGORICAL = [
    "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication", "sick",                          
    "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid", "query_hyperthyroid",            
    "lithium", "goitre", "tumor", "hypopituitary", "psych", "TSH_measured", "T3_measured",                   
    "TT4_measured", "T4U_measured", "FTI_measured", "TBG_measured", "referral_source"               
]
COLS_NUMERICAL = [
    "age","TSH","T3","TT4","T4U","FTI","TBG"
]
COLS_TARGET = "target"

df_train = pd.read_csv(
    paths.TRAIN_CSV,
    header=None,
    names=COLS_ALL,
    na_values=["?"]
)

df_test = pd.read_csv(
    paths.TEST_CSV,
    header=None,
    names=COLS_ALL,
    na_values=["?"]
)

print(df_train.shape)
print(df_test.shape)

display(df_train.head())
display(df_test.head())

# <b><span style='color:#F1A424'>|</span> Visualization</b><a class='anchor' id='visualization'></a> [↑](#top) 

***

Visualization.

In [ ]:
df = df_train.copy(deep=True)

df["target_clean"] = (
    df["target"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.extract(r"(negative|compensated hypothyroid|primary hypothyroid|secondary hypothyroid)", expand=False)
)


# Encode for calculations
df["target_encoded"] = df["target_clean"].map({
    "negative": 0,
    "compensated hypothyroid": 1,
    "primary hypothyroid": 1,
    "secondary hypothyroid": 1
})

df["target_label"] = df["target_encoded"].map({
    0: "Negative",
    1: "Positive"
})

## Pre-Visualization Cleanup

In [ ]:
# Convert ? into missing values
df = df.replace("?", np.nan)


# Clean Gender Column
df["sex"] = (df["sex"].astype("string").str.strip().str.upper())

# Keep only valid M/F values.
# Anything else becomes missing.
df.loc[~df["sex"].isin(["M", "F"]), "sex"] = pd.NA


# Numeric columns
numeric_columns = [
    "age",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI",
    "TBG"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nGender distribution:")
print(df["sex"].value_counts(dropna=False))

## Visualization 1: Class Distribution

In [ ]:
class_counts = (df["target_encoded"].value_counts().reindex([0, 1]))
class_percentages = (class_counts / class_counts.sum() * 100)

labels = ["Negative", "Positive"]
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, class_counts.values)

for bar, count, percentage in zip(bars,class_counts.values, class_percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom"
    )

ax.set_title("Distribution of Target Classes")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Patients")

plt.tight_layout()
plt.show()

# Visualization 2: Positivity rate by Gender

In [ ]:
sex_df = df[df["sex"].isin(["M", "F"]) & df["target_encoded"].notna()].copy()

positive_rate_by_sex = (
    sex_df
    .groupby("sex")["target_encoded"]
    .agg(["mean", "count", "sum"])
    .reset_index()
)
positive_rate_by_sex["positive_rate"] = (positive_rate_by_sex["mean"] * 100)
positive_rate_by_sex["sex_label"] = (
    positive_rate_by_sex["sex"].map({
        "F": "Female",
        "M": "Male"
    })
)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(positive_rate_by_sex["sex_label"], positive_rate_by_sex["positive_rate"])

for bar, rate in zip(bars, positive_rate_by_sex["positive_rate"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{rate:.2f}%",
        ha="center",
        va="bottom"
    )

ax.set_title("Positive Class Rate by Sex")
ax.set_xlabel("Sex")
ax.set_ylabel("Positive Patients (%)")

plt.tight_layout()
plt.show()

## Visualization 3: Distribution by Age

In [ ]:
age_df = df[df["age"].notna() & df["target_encoded"].notna()].copy()

negative_age = age_df.loc[age_df["target_encoded"] == 0, "age"]
positive_age = age_df.loc[age_df["target_encoded"] == 1, "age"]

bins = np.arange(np.floor(age_df["age"].min()), np.ceil(age_df["age"].max()) + 5, 5)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    negative_age,
    bins=bins,
    density=True,
    alpha=0.55,
    label="Negative"
)
ax.hist(
    positive_age,
    bins=bins,
    density=True,
    alpha=0.55,
    label="Positive"
)
ax.axvline(
    negative_age.median(),
    linestyle="--",
    linewidth=2,
    label=f"Negative median = {negative_age.median():.1f}"
)
ax.axvline(
    positive_age.median(),
    linestyle="--",
    linewidth=2,
    label=f"Positive median = {positive_age.median():.1f}"
)

ax.set_title("Age Distribution by Target Class")
ax.set_xlabel("Age")
ax.set_ylabel("Density")
ax.legend()

plt.tight_layout()
plt.show()

## Checking for outliers

In [ ]:
import pandas as pd
import numpy as np

# list of continuous Cols
numeric_cols = [
    "age",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI"
]

# Cecking if all cols are numeric now
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        

# Checking for outliers now
outlier_summary = []

for col in numeric_cols:
    if col not in df.columns:
        continue

    series = df[col].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary.append({
        "feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "num_outliers": len(outliers)
    })
outlier_summary_df = pd.DataFrame(outlier_summary)
outlier_summary_df

## Checking for missing values

In [ ]:
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

# <b><span style='color:#F1A424'>|</span> Data Preprocessing </b><a class='anchor' id='preprocessing'></a> [↑](#top) 

***

Data pre-processing.

### <b><span style='color:#F1A424'>Label Encoding</span></b>

In [ ]:
os.makedirs(paths.OUTPUT, exist_ok=True)

encoders = {}

for col in COLS_CATEGORICAL:
    encoder = LabelEncoder()

    train_values = df_train[col].fillna("Missing").astype(str)
    test_values = df_test[col].fillna("Missing").astype(str)

    encoder.fit(train_values)

    df_train[col] = encoder.transform(train_values)

    mapping = {
        category: idx
        for idx, category in enumerate(encoder.classes_)
    }

    df_test[col] = test_values.map(mapping).fillna(-1).astype(int)

    encoders[col] = encoder

# Save encoders
with open(os.path.join(paths.OUTPUT, "label_encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)

display(df_train.head())
display(df_test.head())

### <b><span style='color:#F1A424'>Cap outliers using IQR</span></b>

In [ ]:
def cap_outliers_iqr(df_train, df_test, columns, factor=1.5):
    bounds = {}

    for col in columns:
        q1 = df_train[col].quantile(0.25)
        q3 = df_train[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - factor * iqr
        upper = q3 + factor * iqr

        df_train[col] = df_train[col].clip(lower, upper)
        df_test[col] = df_test[col].clip(lower, upper)

        bounds[col] = (lower, upper)

    return df_train, df_test, bounds


df_train, df_test, iqr_bounds = cap_outliers_iqr(df_train, df_test, COLS_NUMERICAL)
display(df_train.head())
display(df_test.head())

### <b><span style='color:#F1A424'>Drop column</span></b>

In [ ]:
df_train.drop(columns="TBG", inplace=True)
df_test.drop(columns="TBG", inplace=True)
display(df_train.head())
display(df_test.head())

### <b><span style='color:#F1A424'>Dataset 1 (drop unimportant columns and rows with missing values)</span></b>

In [ ]:
# Dataset 1: drop unimportant columns, then rows with missing values
NUM_COLS_RAW = [c for c in COLS_NUMERICAL if c != "TBG"]

unimportant = [
    c for c in df_train.columns
    if c != "target"
    and (df_train[c].nunique(dropna=True) <= 1 or df_train[c].isna().mean() > 0.5)
]
print("Dropping columns:", unimportant)

df_train_drop = df_train.drop(columns=unimportant).dropna().reset_index(drop=True)
df_test_drop  = df_test.drop(columns=unimportant).dropna().reset_index(drop=True)
NUM_COLS_DROP = [c for c in NUM_COLS_RAW if c in df_train_drop.columns]

print("train_drop:", df_train_drop.shape, " NaNs:", df_train_drop.isna().sum().sum())
print("test_drop: ", df_test_drop.shape,  " NaNs:", df_test_drop.isna().sum().sum())

### <b><span style='color:#F1A424'>Dataset 2 (keep everything and fill missing values with 0)</span></b>

In [ ]:
# Dataset 2: keep all rows and columns, fill missing values with 0
NUM_COLS_RAW = [c for c in COLS_NUMERICAL if c != "TBG"]

df_train_zero = df_train.copy()
df_test_zero  = df_test.copy()
df_train_zero[NUM_COLS_RAW] = df_train_zero[NUM_COLS_RAW].fillna(0)
df_test_zero[NUM_COLS_RAW]  = df_test_zero[NUM_COLS_RAW].fillna(0)

print("train_zero:", df_train_zero.shape, " NaNs:", df_train_zero.isna().sum().sum())
print("test_zero: ", df_test_zero.shape,  " NaNs:", df_test_zero.isna().sum().sum())

### <b><span style='color:#F1A424'>Imputation</span></b>

In [ ]:
NUM_COLS = [c for c in COLS_NUMERICAL if c != "TBG"]
FEATURES = [
    c for c in df_train.columns
    if c not in ("target","patient_id") and df_train[c].nunique() > 1
] #only keep columns that actually have unique values and aren't just constant throughout
NUM_IDX = [FEATURES.index(c) for c in NUM_COLS]

print("NaNs before (train):\n", df_train[NUM_COLS].isna().sum())

low = df_train[FEATURES].min().to_numpy()
high = df_train[FEATURES].max().to_numpy()

#Try a random forest regressor as an imputer
imputer = IterativeImputer(
    estimator = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        n_jobs = -1,
        random_state = 42
    ),
    max_iter=10,
    min_value=low,
    max_value=high,
    random_state=42
)

X_train_imputed = imputer.fit_transform(df_train[FEATURES])
X_test_imputed = imputer.transform(df_test[FEATURES])

df_train[NUM_COLS] = X_train_imputed[:,NUM_IDX]
df_test[NUM_COLS] = X_test_imputed[:,NUM_IDX]



### <b><span style='color:#F1A424'>Scaling</span></b>

In [ ]:
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_test[NUM_COLS] = scaler.transform(df_test[NUM_COLS])


print("NaNs after (train):", df_train[NUM_COLS].isna().sum().sum())
print("NaNs after (test): ", df_test[NUM_COLS].isna().sum().sum())
display(df_train[NUM_COLS].describe().loc[["mean", "std"]])

#Save Imputer and Scaler
with open(os.path.join(paths.OUTPUT, "imputer.pkl"), "wb") as f:
    pickle.dump(imputer, f)
with open(os.path.join(paths.OUTPUT, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

### <b><span style='color:#F1A424'>Transform target</span></b>

In [ ]:
for df in [df_train, df_test]:
    df[["target", "patient_id"]] = df["target"].str.split("|", n=1, expand=True, regex=False)
    df["target"] = df["target"].str.rstrip(".")
    df["patient_id"] = pd.to_numeric(df["patient_id"])

display(df_train.head())
display(df_test.head())

In [ ]:
TARGET_MAPPING = {
    "negative": 0,
    "compensated hypothyroid": 1,
    "primary hypothyroid": 1,
    "secondary hypothyroid": 1
}
print(df_train.target.value_counts()), sep()
print(df_test.target.value_counts()), sep()

df_train["target"] = df_train["target"].map(TARGET_MAPPING)
df_test["target"] = df_test["target"].map(TARGET_MAPPING)

print(df_train.target.value_counts()), sep()
print(df_test.target.value_counts())

### <b><span style='color:#F1A424'>Finish and export Dataset 1 and Dataset 2</span></b>

In [ ]:
# Finish Dataset 1 and Dataset 2: scale + 0/1 target, then export to CSV
def finish_dataset(tr, te, num_cols):
    tr, te = tr.copy(), te.copy()

    sc = StandardScaler()
    tr[num_cols] = sc.fit_transform(tr[num_cols])
    te[num_cols] = sc.transform(te[num_cols])

    for d in (tr, te):
        d[["target", "patient_id"]] = d["target"].str.split("|", n=1, expand=True, regex=False)
        d["target"] = d["target"].str.rstrip(".").map(TARGET_MAPPING)
        d["patient_id"] = pd.to_numeric(d["patient_id"])
        assert d["target"].notna().all(), "Unmapped target label"
        assert d.isna().sum().sum() == 0, "Missing values left"
    return tr, te

train_drop, test_drop = finish_dataset(df_train_drop, df_test_drop, NUM_COLS_DROP)
train_zero, test_zero = finish_dataset(df_train_zero, df_test_zero, NUM_COLS_RAW)

outputs = {
    "hypothyroid_train_drop.csv": train_drop,
    "hypothyroid_test_drop.csv":  test_drop,
    "hypothyroid_train_zero.csv": train_zero,
    "hypothyroid_test_zero.csv":  test_zero,
}
for out_name, out_df in outputs.items():
    out_df.to_csv(os.path.join(paths.OUTPUT, out_name), index=False)
    print(f"Saved {out_name:30s} {out_df.shape}  target={out_df['target'].value_counts().to_dict()}")

## Checking for outliers once again

In [ ]:
outlier_check = []

for i, col in enumerate(NUM_COLS):
    original_lower, original_upper = iqr_bounds[col]
    scaled_lower = (original_lower - scaler.mean_[i]) / scaler.scale_[i]
    scaled_upper = (original_upper - scaler.mean_[i]) / scaler.scale_[i]

    train_outside = ((df_train[col] < scaled_lower) | (df_train[col] > scaled_upper)).sum()
    test_outside = ((df_test[col] < scaled_lower) | (df_test[col] > scaled_upper)).sum()
    outlier_check.append({
        "feature": col,
        "train_outside_bounds": train_outside,
        "test_outside_bounds": test_outside
    })
outlier_check_df = pd.DataFrame(outlier_check)
display(outlier_check_df)

## Checking for missing values once again

In [ ]:
missing_counts = df.isna().sum()
missing_percentages = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentages.round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0]

if missing_summary.empty:
    print("No missing values found.")
else:
    print(missing_summary.sort_values("missing_count", ascending=False))

print("\nTotal missing values:", df.isna().sum().sum())

# Class Distribution

In [ ]:
class_counts = (df_train["target"].value_counts().reindex([0, 1]))
class_percentages = (class_counts / class_counts.sum() * 100)
labels = ["Negative (0)", "Positive (1)"]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, class_counts.values)

for bar, count, percentage in zip(bars, class_counts.values, class_percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_title("Distribution of Target Classes", fontsize=16, fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Patients")
ax.set_ylim(0, class_counts.max() * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## Age Distribution by class

In [ ]:
negative_age = df_train.loc[df_train["target"] == 0, "age"].dropna()
positive_age = df_train.loc[df_train["target"] == 1, "age"].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(negative_age, bins=25, density=True, alpha=0.55, label="Negative (0)")
ax.hist(positive_age, bins=25, density=True, alpha=0.55, label="Positive (1)")

# Medians
negative_median = negative_age.median()
positive_median = positive_age.median()

ax.axvline(negative_median, linestyle="--", linewidth=2, label=f"Negative median = {negative_median:.2f}")
ax.axvline(positive_median, linestyle="--", linewidth=2, label=f"Positive median = {positive_median:.2f}")
ax.set_title("Age Distribution by Target Class", fontsize=16, fontweight="bold")
ax.set_xlabel("Standardized Age")
ax.set_ylabel("Density")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## DF Shapes

In [ ]:
print("Training set shape: ", df_train.shape)
print("Testing set shape: ", df_test.shape)

# <b><span style='color:#F1A424'>|</span> PCA</b><a class='anchor' id='pca'></a> [↑](#top) 

***

Principal Component Analysis: how many components are needed to keep 95% of the variance for each dataset.

In [ ]:
from sklearn.decomposition import PCA

pca_datasets = {
    "imputed": df_train,
    "drop":    train_drop,
    "zero":    train_zero,
}

fig, ax = plt.subplots(figsize=(9, 5))
for ds_name, ds in pca_datasets.items():
    X_pca_input = ds.drop(columns=["target", "patient_id"])
    X_pca_scaled = StandardScaler().fit_transform(X_pca_input)
    pca = PCA().fit(X_pca_scaled)
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n_95 = int(np.argmax(cum_var >= 0.95)) + 1
    print(f"{ds_name:8s} features={X_pca_input.shape[1]}  components for 95% variance = {n_95}")
    ax.plot(range(1, len(cum_var) + 1), cum_var, marker="o", markersize=3, label=ds_name)

ax.axhline(0.95, linestyle="--", color="grey", label="95% variance")
ax.set_title("PCA: Cumulative Explained Variance")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.legend()
plt.tight_layout()
plt.show()

# <b><span style='color:#F1A424'>|</span> Validation</b><a class='anchor' id='validation'></a> [↑](#top) 

***

Validation.

In [ ]:
#5-fold stratified cross-validation on imputed data
X = df_train[FEATURES].to_numpy()
y = df_train["target"].astype(int).to_numpy()
N_SPLITS = 5
RANDOM_STATE = 12345
THRESHOLD = 0.5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# <b><span style='color:#F1A424'>|</span> Modeling</b><a class='anchor' id='modeling'></a> [↑](#top) 

***

Modeling.

### <b><span style='color:#F1A424'>Define models</span></b>

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
}

### <b><span style='color:#F1A424'>CV-Train</span></b>

In [ ]:
mean_fpr = np.linspace(0, 1, 200)

fold_rows = []   # per-fold metrics
roc_data = {}    # per-model ROC curves
oof = {}         # pooled out-of-fold labels / predictions (confusion matrices)

for name, model in models.items():
    tprs, aucs = [], []
    oof_true, oof_pred = [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        model.fit(X[tr_idx], y[tr_idx])
        proba = model.predict_proba(X[va_idx])[:, 1]
        score_pred = (proba >= THRESHOLD).astype(int)

        fold_rows.append({
            "model": name,
            "fold": fold,
            "precision": precision_score(y[va_idx], score_pred, zero_division=0),
            "recall":    recall_score(y[va_idx], score_pred),
            "f1":        f1_score(y[va_idx], score_pred),
            "roc_auc":   roc_auc_score(y[va_idx], proba),
        })

        fpr, tpr, _ = roc_curve(y[va_idx], proba)
        tpr_interp = np.interp(mean_fpr, fpr, tpr)
        tpr_interp[0] = 0.0
        tprs.append(tpr_interp)
        aucs.append(roc_auc_score(y[va_idx], proba))

        oof_true.extend(y[va_idx])
        oof_pred.extend(proba)

    roc_data[name] = (tprs, aucs)
    oof[name] = (np.array(oof_true), np.array(oof_pred))

results = pd.DataFrame(fold_rows)
metrics = ["precision", "recall", "f1", "roc_auc"]

### <b><span style='color:#F1A424'>Full train</span></b>

For model interpretability

In [ ]:
from sklearn.base import clone

full_models = {}

for name, model in models.items():
    full_models[name] = clone(model)
    full_models[name].fit(X, y)

# <b><span style='color:#F1A424'>|</span> Results</b><a class='anchor' id='results'></a> [↑](#top) 

***

Results.

In [ ]:
# Average performance per model (mean ± std across the 5 folds)
mean_ = results.groupby("model")[metrics].mean()
std_ = results.groupby("model")[metrics].std()

summary = mean_.copy().astype(object)
for m in metrics:
    summary[m] = [f"{mean_.loc[n, m]:.3f} ± {std_.loc[n, m]:.3f}" for n in mean_.index]

display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, (tprs, aucs) in roc_data.items():
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, lw=2.5,
            label=f"{name} (AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f})")
    ax.fill_between(mean_fpr,
                    np.clip(mean_tpr - std_tpr, 0, 1),
                    np.clip(mean_tpr + std_tpr, 0, 1),
                    alpha=0.15)

ax.plot([0, 1], [0, 1], "--", color="red", lw=1, label="Random Guess")
ax.set_title("Mean ROC Curve (5-Fold Stratified CV)", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
means = results.groupby("model")[metrics].mean()
stds = results.groupby("model")[metrics].std()

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics))
w = 0.35

for i, name in enumerate(means.index):
    bars = ax.bar(x + i * w - w / 2, means.loc[name], w,
                  yerr=stds.loc[name], capsize=4, label=name)
    for b, v in zip(bars, means.loc[name]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}",
                ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(["Precision", "Recall", "F1", "ROC-AUC"])
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("5-Fold Stratified CV: Model Comparison", fontsize=14, fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
threshold = 0.5

for ax, (name, (yt, yp)) in zip(axes, oof.items()):
    yp = (yp >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        yt, yp, display_labels=["Negative", "Positive"],
        cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name, fontweight="bold")

plt.tight_layout()
plt.show()

# <b><span style='color:#F1A424'>|</span> Ensembling</b><a class='anchor' id='ensembling'></a> [↑](#top) 

***

Ensembling.

In [ ]:
dfs = []
for k,v in oof.items():
    name = "_".join(k.lower().split())
    probas = oof[k][1]
    aux = pd.DataFrame(probas)
    aux.columns = [name]
    dfs.append(aux)

oof_df = pd.concat(dfs, axis=1)
oof_df["y_true"] = yt
oof_df.head()

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
cols = [x for x in oof_df.columns if x != "y_true"]
meta_model.fit(oof_df[cols], oof_df["y_true"])
weights = meta_model.coef_
intercept = meta_model.intercept_

# <b><span style='color:#F1A424'>|</span> Interpretability</b><a class='anchor' id='interpretability'></a> [↑](#top) 

***

In logistic regression, the log-odds of the positive class are modeled as a linear combination of the input features:

$$
\ln\left(\frac{p}{1-p}\right)
= \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n
$$

Equivalently, we can express the log-odds using base 10:

$$
\log_{10}\left(\frac{p}{1-p}\right)
= \frac{\beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n}{\ln(10)}
$$

Since scikit-learn uses the natural logarithm, exponentiating both sides of the original equation yields:

$$
\frac{p}{1-p} = e^{\beta_0 + \beta_1 x_1 + \cdots + \beta_n x_n}
$$

Notice that exponentiating the log-odds yields the **odds**, not the probability.

To understand the contribution of a particular feature $x_i$, we can set the remaining features to zero:

$$
\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_i x_i
$$

Increasing $x_i$ by one unit multiplies the odds by:

$$
\text{Odds ratio}
= \frac{e^{\beta_0 + \beta_i(x_i+1)}}{e^{\beta_0 + \beta_i x_i}}
= e^{\beta_i}
$$

Therefore, each coefficient $\beta_i$ can be converted into an **odds ratio** by exponentiating it:

$$
\text{Odds ratio}=e^{\beta_i}
$$

This tells us by what factor the odds of $Y=1$ are multiplied when the corresponding feature increases by one unit, holding all other features constant. Setting the other features to zero is useful for illustration, but is not necessary for this interpretation to hold.

## **1. Odds ratios**

| Feature         | Coefficient | Odds ratio $e^{\beta_i}$ |
| --------------- | ----------- | ------------------------ |
| TSH             | 4.305790    | 74.13                    |
| on_thyroxine    | −3.776397   | 0.0229                   |
| thyroid_surgery | −3.023538   | 0.0486                   |
| TT4             | −2.117561   | 0.1203                   |
| TSH_measured    | 1.470737    | 4.35                     |

## **2. Interpretation of each feature**

Assuming $Y=1$ represents the presence of the thyroid condition we are predicting, and the indicator variables are encoded as 0 or 1:

- **TSH:** Increasing TSH by one unit multiplies the odds of having the condition by **74.13**.
- **on_thyroxine:** Patients taking thyroxine have **97.71% lower odds** of having the condition than patients not taking thyroxine.
- **thyroid_surgery:** Patients who have undergone thyroid surgery have **95.14% lower odds** of having the condition than patients who have not.
- **TT4:** Increasing TT4 by one unit decreases the odds of having the condition by **87.97%**.
- **TSH_measured:** Patients whose TSH was measured have **4.35 times the odds** of having the condition compared to patients whose TSH was not measured.

All these interpretations assume that the remaining features are held constant.

If continuous features were standardized before training, a one-unit increase corresponds to an increase of one standard deviation in the original feature.

## **3. Interpretation of the intercept**

The intercept of our logistic regression model is:

$$
\beta_0 = -7.76041576
$$

The intercept represents the log-odds of $Y=1$ when all input features are equal to zero:

$$
\ln\left(\frac{p}{1-p}\right) = -7.76041576
$$

Exponentiating both sides, we obtain the baseline odds:

$$
\frac{p}{1-p} = e^{-7.76041576} \approx 0.000426
$$

This means that the baseline odds of having the condition are approximately **1 to 2346**.

To obtain the baseline probability, we isolate $p$:

$$
p = \frac{e^{-7.76041576}}{1+e^{-7.76041576}}
\approx 0.000426
$$

Therefore, when all input features are zero, the model predicts a probability of having the condition of approximately **0.0426%**.

If the features were standardized, zero corresponds to their original mean values rather than their original zero values. Furthermore, the intercept should not necessarily be interpreted as the prevalence of the condition in the population, since it represents the prediction for a specific feature configuration that may not correspond to a realistic patient.

In [ ]:
model = full_models["Logistic Regression"]

coef = pd.Series(model.coef_[0], index=FEATURES)
coef = coef.sort_values(key=abs, ascending=False)
print(f"Intercept: {model.intercept_[0]}")
coef.plot(
    kind="barh",
    figsize=(10, 6),
    color=["steelblue" if v >= 0 else "indianred" for v in coef]
)

plt.gca().invert_yaxis()
plt.xlabel("Coefficient")
plt.title("Logistic Regression - Feature Importance")
plt.axvline(0, color="black")
plt.show()

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X, y)
importances = xgb_model.feature_importances_
gain_scores = xgb_model.get_booster().get_score(importance_type='gain')
importance = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURES
).sort_values()
importance.plot(kind="barh", figsize=(10, 6))
plt.title("XGBoost - Feature Importance")
plt.xlabel("Importance")
plt.show()